# TSV-grounded CPW · Driven Terminal Wave model inspection

Build a finite CPW whose top and bottom Ground planes are stitched by
two rows of hollow TSVs. The Signal reaches both die X edges so the
exterior cuts carry Wave ports. This 3D example makes the TSV
distance/placement and Ground paths inspectable; it does not predict
their optimum density, impedance or slotline suppression. A Q2D slice
cannot represent the discrete TSV array. See the [TSV
guide](../../../../simulation/tsv-model-inspection.qmd) and [Driven
Terminal
requirement](../../../../simulation/requirements/driven-terminal.qmd).

Use this OrPen source candidate with the fixed [SCGSim
checkpoint](https://github.com/OrPenStrike/scgsim/tree/c9270c7e76e0b2df187dddde462079e00d410dd0)
in an isolated Python 3.12/3.13 runtime. The explicit native stage uses
AEDT 2024 R2 / PyAEDT 1.3.0. No ordinary environment/pin update is
performed here.

## Setup and Imports

The following imports are public source/model APIs; they do not open a
Desktop.

## Setup and Run Controls

In [ ]:
import hashlib
import subprocess
import sys
from dataclasses import replace
from importlib.metadata import version
from pathlib import Path
from uuid import uuid4

from IPython.display import display
from scgsim.aedt import (
    TerminalPort,
    prepare_hfss_driven_from_geometry,
    prepare_planar_geometry_input,
    resolve_prepared_model,
)
from scgsim.geometry import GeometryBuildInput, LayoutPolygonSpec, SemanticEntitySpec

import orpen_sc_pdk
from orpen_sc_pdk.cells import get_tsv_source_records, tsv_ground_stitched_cpw

WORKFLOW_ACTION = "source"  # source | prepare_handoff | build_model | inspect_model
DIE = "D0"
WIDTH_UM = 10.0
GAP_UM = 6.0
DIE_SIDE_UM = 1000.0
BORE_DIAMETER_UM = 60.0
WALL_THICKNESS_UM = 0.2
TSV_ROW_Y_UM = 100.0
TSV_X_UM = (-300.0, -100.0, 100.0, 300.0)
MODEL = "thin_film"
RUN_CONTROL = None
RUN_ID = f"tsv-cpw-terminal-{uuid4().hex[:12]}"
OUTPUT_ROOT = Path(".artifacts/SCGSimExamples/TSV")
RUN_DIR = OUTPUT_ROOT / RUN_ID
SOURCE_DIR = OUTPUT_ROOT / f"{RUN_ID}-source"
RETURNED_RUN_DIR = None  # Set the explicit actual returned run for inspect_model.
PROJECT_NAME = "TSVGroundCPWTerminalThinFilmModelOnly"
DESIGN_NAME = "Terminal"
REGION_PADDING_XYZ_UM = (0.0, 100.0, 1000.0)
GEOMETRY_WORKERS = 1
RESOURCES = None  # No solver HPC request; model-process allocation is arranged separately.
SIGNAL_ENTITY_IDS = (f"{DIE}/SIGNAL",)
GROUND_ENTITY_IDS = (f"{DIE}/TOP_GROUND", f"{DIE}/BACK_GROUND") + tuple(
    f"{DIE}/TSV_{i}" for i in range(2 * len(TSV_X_UM))
)
PORT_ORDER = ((1, "o1", "-X"), (2, "o2", "+X"))
DEEMBED_UM = 0.0
orpen_sc_pdk.activate()
display({name: version(name) for name in ("orpen-sc-pdk", "gdsfactory", "scgsim")})

`source` only builds the layout and source records. `prepare_handoff`
constructs the backend package; `build_model` opens an owned Desktop for
a no-analysis model transaction. `inspect_model` is file-only
returned-run reading. Resource flags do not contain CAD descendants or
prove an aggregate OS cap.

## Create Simulation Component / Coupon

In [ ]:
if WORKFLOW_ACTION != "inspect_model":
    coupon = tsv_ground_stitched_cpw(
        die=DIE,
        width_um=WIDTH_UM,
        gap_um=GAP_UM,
        die_side_um=DIE_SIDE_UM,
        bore_diameter_um=BORE_DIAMETER_UM,
        wall_thickness_um=WALL_THICKNESS_UM,
        tsv_row_y_um=TSV_ROW_Y_UM,
        tsv_x_um=TSV_X_UM,
    )
    coupon.plot()

The source places eight hollow TiN TSVs outside the two CPW gaps. Its
top and back Ground remain positive conductor regions with explicitly
preserved holes. Surface Al thickness and substrate Z come from the PDK,
not the port settings.

## Initialize AEDT Project / App

The explicit `build_model` CLI owns its HFSS Driven Terminal session. No
source cell attaches to a live Desktop or manually creates another
native client.

## Import GDS and Build the HFSS/Q3D/Q2D Model

The canonical OrPen getter supplies actual placed polygon exteriors,
holes, source occurrences, physical levels and materials. This cell only
adopts the explicit case Net grouping and constructs SCGSim’s public
input records. It does not infer connectivity or register invented
children. This cell constructs source input; actual native import/build
belongs to the later CLI transaction, not this source observation.

In [ ]:
if WORKFLOW_ACTION != "inspect_model":
    bounds = coupon.dbbox()
    regions = coupon.info["tsv_source"]["source_regions"]
    source_bottom = min(r["z_interval_um"][0] for r in regions)
    source_top = max(r["z_interval_um"][1] for r in regions)
    px, py, pz = REGION_PADDING_XYZ_UM
    background = {
        "semantic_id": "CALLER_BACKGROUND",
        "role": "solution_region",
        "material_id": "vacuum",
        "material_kind": "vacuum",
        "priority": 99,
        "geometry_kind": "domain",
        "geometry": {
            "domain_bounds_um": {
                "x_min_um": bounds.left - px,
                "x_max_um": bounds.right + px,
                "y_min_um": bounds.bottom - py,
                "y_max_um": bounds.top + py,
            },
            "z_min_um": source_bottom - pz,
            "z_max_um": source_top + pz,
        },
        "metadata": {"meaning": "computational background complement"},
    }
    source = get_tsv_source_records(coupon, host_solution_region=background)
    caller_nets = {
        **dict.fromkeys(SIGNAL_ENTITY_IDS, "Signal"),
        **dict.fromkeys(GROUND_ENTITY_IDS, "Ground"),
    }
    records = source["geometry_input"]
    stack = source["stack"]
    for layer in stack["layers"]:
        layer["net_id"] = caller_nets[layer["semantic_id"]]
    # Keep physical material records; add only the compiler's kind vocabulary.
    geometry_kinds = {
        "superconductor": "conductor",
        "dielectric": "dielectric",
        "vacuum": "vacuum",
    }
    stack["materials"] = {
        name: {
            **record,
            "kind": geometry_kinds[record["material_kind"]],
            "permittivity": record["relative_permittivity"],
        }
        for name, record in stack["materials"].items()
    }
    build_input = GeometryBuildInput(
        polygons=tuple(LayoutPolygonSpec(**p) for p in records["polygons"]),
        entities=tuple(
            SemanticEntitySpec(**{**e, "net_id": caller_nets.get(e["semantic_id"])})
            for e in records["entities"]
        ),
        solution_regions=records["solution_regions"],
        metadata={**records["metadata"], "caller_final_nets": caller_nets},
    )
    display(
        {
            "source_parameters": coupon.info["tsv_source"]["parameters"],
            "source_contacts": records["metadata"]["source_contacts"],
            "source_ports": records["metadata"]["source_ports"],
            "caller_nets": caller_nets,
            "modeling_declarations": {
                e.semantic_id: e.metadata.get("aedt_modeling")
                for e in build_input.entities
                if e.material_kind == "conductor"
            },
        }
    )

## Geometry Verification

These are source observations. Silicon already has the authored bore
holes; the TSV wall is an annulus, not a filled cylinder. In
`thin_film`, face films follow their declared contact side while TSV
`retained_solid` levels stay finite. SCGSim owns the effective-Z mapping
and the actual backend geometry. The current AEDT path consumes sampled
GDS polygon witnesses; exact circular recipes remain source intent, not
active native curve reconstruction.

### Fresh source witness

In [ ]:
if WORKFLOW_ACTION in {"prepare_handoff", "build_model"}:
    SOURCE_DIR.mkdir(parents=True, exist_ok=False)
    gds_path = coupon.write_gds(SOURCE_DIR / "source.gds", with_metadata=False)
    gds_bytes = Path(gds_path).read_bytes()
    gds_sha256 = hashlib.sha256(gds_bytes).hexdigest()
    stack["metadata"]["gds_sha256"] = gds_sha256
    build_input = replace(build_input, metadata={**build_input.metadata, "gds_sha256": gds_sha256})

The exported witness and normalized source records come from the same
component. The backend helper owns actual lowering; this notebook does
not interpret a source GDS hash as an actual native geometry
observation.

## Materials and Boundaries

Al and TiN remain the physical source materials; their PDK
superconducting declarations select ideal native conductor treatment.
Silicon is a perforated dielectric solid. The caller-authored background
is the surrounding complement, not a second overlapping occupied body.
Final Nets are the explicit grouping above, not a connectivity result
inferred by the notebook.

## Simulation Setup

`RUN_CONTROL = None` requests model-only preparation. No frequency,
modes, adaptive setup or sweep is silently filled in. A later physics
calculation needs its own explicitly authored setup and operation
assignment.

## Simulation Configuration

The outward cut normals are −X / +X, distinct from the source
Signal-to-Ground direction. Ports use explicit source Entity IDs rather
than guessed native object names. This checkpoint’s Wave record does not
carry a caller integration line; source direction facts are not
presented as an independently assigned native line. Renormalization is
off; zero de-embed distance is not a reference plane shift. Inactive
native 50 Ω metadata is not a source load or solved Zo.

In [ ]:
HANDOFF = None
if WORKFLOW_ACTION in {"prepare_handoff", "build_model"}:
    ports = tuple(
        TerminalPort(
            index=index,
            name=name,
            side=side,
            reference_objects=GROUND_ENTITY_IDS,
            signal_entity_ids=SIGNAL_ENTITY_IDS,
            deembed_um=DEEMBED_UM,
        )
        for index, name, side in PORT_ORDER
    )
    geometry = prepare_planar_geometry_input(build_input, prepared_stack=stack, modeling=MODEL)
    HANDOFF = prepare_hfss_driven_from_geometry(
        modeling=MODEL,
        geometry=geometry,
        project_name=PROJECT_NAME,
        design_name=DESIGN_NAME,
        run_control=RUN_CONTROL,
        ports=ports,
        output_dir=RUN_DIR,
        geometry_workers=GEOMETRY_WORKERS,
        resources=RESOURCES,
    )
    display([port.to_payload() for port in ports])
    display(HANDOFF)

Zero X enclosure padding makes the Wave cuts exterior. The remaining
Region faces retain the closed HFSS exterior treatment, without
Radiation or PML. `run_control=None` means no adaptive frequency,
Analysis Setup or sweep.

## Solve and Export

**Model-only: no solve.** The explicit action builds, reads, saves and
releases the model without Analyze or a sweep.

In [ ]:
if WORKFLOW_ACTION == "build_model":
    subprocess.run(
        [
            sys.executable,
            "-m",
            "scgsim.aedt.run",
            "--handoff",
            str(HANDOFF.metadata_path),
            "--execute",
            "--prepare-only",
        ],
        check=True,
    )

## Adaptive-Pass Convergence / Solver Diagnostics

No adaptive passes, delta-S or sweep progress exist in this stage.
Retain native preparation failures without an automatic retry or
substituted result.

## Results: Plots and Readable Tables

### Physics Analysis Results

Inspect actual Wave type, port/terminal order and available
reference-object readback, alongside face sheets, solid TSV walls and
cut Silicon. Preserve raw boundary-query missingness; an empty query
must not be replaced by a guessed port result. No S-parameters, Port Zo
or propagation result is expected.

In [ ]:
PREPARED_MODEL = None
if WORKFLOW_ACTION == "inspect_model":
    PREPARED_MODEL = resolve_prepared_model(RETURNED_RUN_DIR)
    display(PREPARED_MODEL.project_path)
    display(PREPARED_MODEL.readback)

### Simulation Performance / Benchmarks

Retain actual build/save/release evidence, without adaptive passes or
solved benchmarks. A view of the model is not a field, connectivity or
impedance result.

In [ ]:
if PREPARED_MODEL is not None:
    display({"mode": PREPARED_MODEL.mode, "model_stage": PREPARED_MODEL.model_stage})

## Save and Release AEDT

The transaction saves and releases its own Desktop. Keep each project
with its necessary unsolved companions and inspect a separate writable
copy. Existing Solid projects and images remain their original evidence,
not outputs of this mixed Thin Film/Solid source notebook.